In [20]:
import json
import numpy as np
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()
RUNS_DIR = PROJECT_ROOT / "runs" / "test"
FINAL_DIR = PROJECT_ROOT / "resultsv2" / "final"
BENCH_DIR = PROJECT_ROOT / "resultsv2" / "latency_bench"

bench_std = {}
for bench_file in BENCH_DIR.glob("*.json"):
    with open(bench_file) as f:
        bd = json.load(f)
    bench_std[bd["run_id"]] = round(np.std(bd["latencies_ms"]), 3)

latency_std_map = {
    ("pytorch", "fp32"): bench_std["pytorch_fp32_in8b_cuda_bs1"],
    ("pytorch", "fp16"): bench_std["pytorch_fp32_in8b_cuda_bs1"],
    ("tensorrt", "fp32"): bench_std["tensorrt_fp32_in4b_cuda_bs1"],
    ("tensorrt", "fp16"): bench_std["tensorrt_fp32_in4b_cuda_bs1"],
    ("tensorrt", "fp8"): bench_std["tensorrt_int8_in8b_cuda_bs1"],
    ("tensorrt", "int8"): bench_std["tensorrt_int8_in8b_cuda_bs1"],
    ("tensorrt", "int4"): bench_std["tensorrt_int8_in8b_cuda_bs1"],
}

rows = []
for result_file in sorted(RUNS_DIR.rglob("result.json")):
    with open(result_file) as f:
        data = json.load(f)

    parts = result_file.relative_to(RUNS_DIR).parts
    method = parts[0]
    seed = parts[1]
    run_name = parts[2]

    cfg = data["config"]
    extra = data.get("config_extra", {})
    res = data["results"]

    if method == "qat":
        backend = "tensorrt" if "trt" in run_name else "pytorch"
        precision = extra.get("qat_precision", cfg.get("model_precision", ""))
        if backend == "tensorrt":
            precision = "qat_int8"
    else:
        backend = cfg.get("backend", "tensorrt" if "trt" in run_name else "pytorch")
        precision = cfg.get("model_precision", "")

    input_bits = extra.get("input_quant_bits", cfg.get("input_quant_bits"))

    row = {
        "method": method,
        "backend": backend,
        "precision": precision,
        "input_bits": int(input_bits),
        "seed": seed,
        "run_id": data.get("run_id", ""),
        "top1": round(res["top1_acc"], 2),
        "top5": round(res["top5_acc"], 2),
        "latency_ms": round(res["infer_ms_avg"], 3),
        "throughput": round(res.get("throughput_infer_sps", res.get("throughput_sps", 0)), 1),
        "total_samples": res["total_batches"],
    }

    if method == "ptq":
        row["latency_std"] = latency_std_map.get((backend, precision), None)

    rows.append(row)

df = pd.DataFrame(rows).sort_values(
    ["method", "seed", "backend", "input_bits", "precision"]
).reset_index(drop=True)

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", None)

df_ptq = df[df["method"] == "ptq"].reset_index(drop=True)
df_ptq.to_csv(FINAL_DIR / "raw_ptq.csv", index=False)
df_ptq

,method,backend,precision,input_bits,seed,run_id,top1,top5,latency_ms,throughput,total_samples,latency_std
0,ptq,pytorch,fp16,1,seed_1,torch_ptq_fp16_b1_cuda,69.72,89.38,3.006,332.7,4970,1.244
1,ptq,pytorch,fp32,1,seed_1,torch_ptq_fp32_b1_cuda,69.68,89.38,2.968,337.0,4970,1.244
2,ptq,pytorch,fp16,2,seed_1,torch_ptq_fp16_b2_cuda,76.94,93.08,3.063,326.5,4970,1.244
3,ptq,pytorch,fp32,2,seed_1,torch_ptq_fp32_b2_cuda,76.88,93.08,2.968,337.0,4970,1.244
4,ptq,pytorch,fp16,4,seed_1,torch_ptq_fp16_b4_cuda,78.41,93.64,3.054,327.5,4970,1.244
5,ptq,pytorch,fp32,4,seed_1,torch_ptq_fp32_b4_cuda,78.43,93.68,3.102,322.3,4970,1.244
6,ptq,pytorch,fp16,8,seed_1,torch_ptq_fp16_b8_cuda,77.87,93.38,3.042,328.7,4970,1.244
7,ptq,pytorch,fp32,8,seed_1,torch_ptq_fp32_b8_cuda,77.85,93.40,3.074,325.4,4970,1.244
8,ptq,tensorrt,fp16,1,seed_1,trt_ptq_fp16_b1_cuda,69.68,89.38,0.477,2097.8,4970,0.136
9,ptq,tensorrt,fp32,1,seed_1,trt_ptq_fp32_b1_cuda,69.70,89.38,0.949,1054.2,4970,0.136


## Final results — PTQ & QAT

Load all test runs, split into PTQ and QAT tables, average across seeds, and save to `resultsv2/final/`. Latency std comes from the dedicated 10k-iteration benchmarks in `resultsv2/latency_bench/`.

In [21]:
df_qat = df[df["method"] == "qat"].sort_values(
    ["backend", "seed", "precision"]
).reset_index(drop=True)
df_qat.to_csv(FINAL_DIR / "raw_qat.csv", index=False)
df_qat

,method,backend,precision,input_bits,seed,run_id,top1,top5,latency_ms,throughput,total_samples,latency_std
0,qat,pytorch,int4,1,seed_1,torch_qat_int4_b1_cuda,69.18,89.11,10.097,99.0,4970,NaN
1,qat,pytorch,int4,2,seed_1,torch_qat_int4_b2_cuda,76.28,92.54,10.236,97.7,4970,NaN
2,qat,pytorch,int4,4,seed_1,torch_qat_int4_b4_cuda,77.89,93.28,11.201,89.3,4970,NaN
3,qat,pytorch,int4,8,seed_1,torch_qat_int4_b8_cuda,77.22,93.32,10.195,98.1,4970,NaN
4,qat,pytorch,int8,1,seed_1,torch_qat_int8_b1_cuda,69.56,89.54,9.335,107.1,4970,NaN
5,qat,pytorch,int8,2,seed_1,torch_qat_int8_b2_cuda,76.64,92.86,9.120,109.7,4970,NaN
6,qat,pytorch,int8,4,seed_1,torch_qat_int8_b4_cuda,78.31,93.62,8.969,111.5,4970,NaN
7,qat,pytorch,int8,8,seed_1,torch_qat_int8_b8_cuda,77.85,93.46,9.063,110.3,4970,NaN
8,qat,pytorch,int4,1,seed_2,torch_qat_int4_b1_cuda,69.62,88.89,10.641,94.0,4970,NaN
9,qat,pytorch,int4,2,seed_2,torch_qat_int4_b2_cuda,76.16,93.16,10.311,97.0,4970,NaN


In [22]:
avg_ptq = df_ptq.groupby(["method", "backend", "input_bits", "precision"]).agg(
    top1=("top1", "mean"),
    top5=("top5", "mean"),
    latency_ms=("latency_ms", "mean"),
    latency_std=("latency_std", "first"),
    throughput=("throughput", "mean"),
).round(3).reset_index()
avg_ptq.to_csv(FINAL_DIR / "avg_ptq.csv", index=False)
avg_ptq

,method,backend,input_bits,precision,top1,top5,latency_ms,latency_std,throughput
0,ptq,pytorch,1,fp16,69.550,89.480,3.065,1.244,326.333
1,ptq,pytorch,1,fp32,69.550,89.473,2.980,1.244,335.567
2,ptq,pytorch,2,fp16,76.887,93.140,3.075,1.244,325.233
3,ptq,pytorch,2,fp32,76.860,93.140,2.984,1.244,335.133
4,ptq,pytorch,4,fp16,78.330,93.640,3.087,1.244,323.967
5,ptq,pytorch,4,fp32,78.337,93.653,3.082,1.244,324.467
6,ptq,pytorch,8,fp16,77.977,93.580,3.048,1.244,328.100
7,ptq,pytorch,8,fp32,77.990,93.580,3.043,1.244,328.767
8,ptq,tensorrt,1,fp16,69.557,89.473,0.475,0.136,2105.500
9,ptq,tensorrt,1,fp32,69.557,89.473,0.941,0.136,1062.933


In [24]:
avg_qat = df_qat.groupby(["method", "backend", "input_bits", "precision"]).agg(
    top1=("top1", "mean"),
    top5=("top5", "mean"),
    latency_ms=("latency_ms", "mean"),
    throughput=("throughput", "mean"),
).round(3).reset_index().sort_values(
    ["backend", "precision", "input_bits"]
).reset_index(drop=True)
avg_qat.to_csv(FINAL_DIR / "avg_qat.csv", index=False)
avg_qat

,method,backend,input_bits,precision,top1,top5,latency_ms,throughput
0,qat,pytorch,1,int4,69.223,89.073,10.252,97.600
1,qat,pytorch,2,int4,76.120,92.717,10.180,98.267
2,qat,pytorch,4,int4,77.627,93.300,10.334,97.267
3,qat,pytorch,8,int4,77.253,93.220,10.022,100.000
4,qat,pytorch,1,int8,69.350,89.460,8.778,114.267
5,qat,pytorch,2,int8,76.713,93.100,8.938,112.533
6,qat,pytorch,4,int8,78.417,93.473,8.732,114.733
7,qat,pytorch,8,int8,77.997,93.373,8.738,114.533
8,qat,tensorrt,1,qat_int8,71.490,94.680,0.761,1314.100
9,qat,tensorrt,2,qat_int8,81.060,96.600,0.698,1431.700
